# 03 — Modeling

**Job:** train, validate honestly, interpret, and persist the classifier.

**Contract:** reads `data/processed/feature_matrix.parquet` → writes
`model.joblib`.

Because we start from the processed checkpoint, model iteration is fast: tweak
a hyperparameter and re-run only this notebook — no re-parsing, no
re-feature-building.

## 01 — Imports and paths

In [ ]:
"""01_data_exploration.ipynb — Preliminary exploration of ClinVar missense variants."""
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GroupShuffleSplit,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
)
import joblib

from load_clinvar import load_and_label_clinvar, print_funnel

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_IN = DATA_DIR / "processed" / "feature_matrix.parquet"
MODEL_OUT = PROJECT_ROOT / "model.joblib"

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 02 — Load the feature matrix

In [ ]:
X_df = pd.read_parquet(PROCESSED_IN)
feature_cols = [c for c in X_df.columns if c not in ("label", "gene")]
X = X_df[feature_cols].values
y = X_df["label"].values
print(f"Loaded {X_df.shape[0]:,} variants × {len(feature_cols)} features")
print("Features:", feature_cols)

## 03 — Stratified train / test split

20% held out and never seen during training. `stratify=y` preserves the
class balance in both splits, so the test estimate is comparable to training
conditions on this imbalanced data.

In [ ]:
# %%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(y_train):,}  Test: {len(y_test):,}  "
      f"(test pathogenic frac: {y_test.mean():.1%})")

## 04 — Model A: Logistic Regression (interpretable baseline)

`StandardScaler` lives *inside* the pipeline so it is fit on training data
only within each fold — no leakage. `class_weight="balanced"` counters the
pathogenic-heavy imbalance. Standardized coefficients are directly comparable:
sign = direction, magnitude = strength.

In [ ]:
# %%
logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced",
                       random_state=RANDOM_STATE),
)
logreg.fit(X_train, y_train)
p_lr = logreg.predict_proba(X_test)[:, 1]

In [ ]:
print("LOGISTIC REGRESSION")
print(f"  ROC-AUC: {roc_auc_score(y_test, p_lr):.3f}")
print(f"  PR-AUC : {average_precision_score(y_test, p_lr):.3f}")

In [ ]:
coefs = pd.Series(
    logreg.named_steps["logisticregression"].coef_[0], index=feature_cols
).sort_values()
print("\nLR coefficients (standardized; + => pushes toward pathogenic):")
print(coefs.to_string())

## 05 — Model B: Hist Gradient Boosting (the workhorse)

No scaler: trees split on thresholds and are invariant to feature scale. The
hyperparameters are all regularization knobs (shallow trees, low learning
rate, L2) guarding against overfitting.

In [ ]:
# %%
hgb = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_depth=4,
    l2_regularization=1.0, random_state=RANDOM_STATE,
)
hgb.fit(X_train, y_train)
p_hgb = hgb.predict_proba(X_test)[:, 1]

## 06 — 5-fold stratified CV (don't trust a single split)

Report mean ± std: the mean is the performance estimate, the std is the
uncertainty around it. A tight band means a stable model.

In [ ]:
# %%
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, model in [("LogReg", logreg), ("HistGB", hgb)]:
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    print(f"{name}: ROC-AUC {scores.mean():.3f} ± {scores.std():.3f}  "
          f"{np.round(scores, 3)}")

## 07 — Confusion matrix + classification report

At a 0.5 threshold, on the better model by test ROC-AUC. The per-class report
exposes minority-class recall that aggregate accuracy would hide.

In [ ]:
# %%
best_p = p_hgb if roc_auc_score(y_test, p_hgb) >= roc_auc_score(y_test, p_lr) else p_lr
best_name = "HistGB" if best_p is p_hgb else "LogReg"
y_pred = (best_p >= 0.5).astype(int)

In [ ]:
print(f"Best model: {best_name}")
print("\nConfusion matrix [rows=true, cols=pred] (0=benign,1=pathogenic):")
print(confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred,
                                   target_names=["benign", "pathogenic"]))

## 08 — ROC + PR curves

ROC summarizes ranking quality across all thresholds; PR is the honest view on
imbalanced data (no large true-negative cushion to flatter the score).

In [ ]:
# %%
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for name, p in [("LogReg", p_lr), ("HistGB", p_hgb)]:
    fpr, tpr, _ = roc_curve(y_test, p)
    ax1.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, p):.3f})")
ax1.plot([0, 1], [0, 1], "k--", lw=0.8)
ax1.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC")
ax1.legend()
for name, p in [("LogReg", p_lr), ("HistGB", p_hgb)]:
    prec, rec, _ = precision_recall_curve(y_test, p)
    ax2.plot(rec, prec, label=f"{name} (AP={average_precision_score(y_test, p):.3f})")
ax2.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall")
ax2.legend()
plt.tight_layout()
plt.show()

## 09 — Permutation importance (model-agnostic, on held-out data)

How much does ROC-AUC drop when each feature is shuffled? Measured on the test
set, so it reflects genuine reliance — unbiased unlike a tree's built-in
impurity importances. Expect `blosum62` to dominate.

In [ ]:
r = permutation_importance(hgb, X_test, y_test, n_repeats=10,
                           random_state=RANDOM_STATE, scoring="roc_auc")
imp = pd.Series(r.importances_mean, index=feature_cols).sort_values(ascending=False)
print("Permutation importance (drop in ROC-AUC when feature shuffled):")
print(imp.to_string())

## 10 — Gene-held-out validation (beware gene leakage)

Random splits let the same gene sit in train *and* test, so the model can
memorize "gene X tends to be pathogenic" instead of learning substitution
chemistry. Holding entire genes out is the honest test. A large drop vs. the
Cell 6 random-split CV reveals leakage.

In [ ]:
groups = X_df["gene"].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(X, y, groups))
hgb_g = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_depth=4,
    l2_regularization=1.0, random_state=RANDOM_STATE,
).fit(X[tr_idx], y[tr_idx])
p_g = hgb_g.predict_proba(X[te_idx])[:, 1]
print(f"Gene-held-out ROC-AUC: {roc_auc_score(y[te_idx], p_g):.3f}  "
      f"(compare to the random-split CV in Cell 6)")

## 11 — Persist the model for the FastAPI service

Save the fitted model *with* `feature_cols`: at inference time features must
be fed in the exact training order. Bundling the column list prevents a
silent, catastrophic misalignment.

In [ ]:
joblib.dump({"model": hgb, "feature_cols": feature_cols}, MODEL_OUT)
print(f"Saved {MODEL_OUT.relative_to(PROJECT_ROOT)}")


## Cell 12 — Threshold analysis (the clinical tradeoff)

At the default 0.5 threshold the model has high benign recall but only ~0.42
pathogenic recall — it misses more than half of pathogenic variants. For a
clinical screening tool the costly error is a MISSED pathogenic call, so 0.5
(an arbitrary cutoff on a 7:1-imbalanced problem) is the wrong operating
point. Here we sweep thresholds and expose the precision we trade for recall,
rather than hard-coding 0.5.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, average_precision_score
 
# Use the better model's held-out probabilities (p_hgb from Cell 5).
probs = p_hgb
 
print(f"HistGB PR-AUC (threshold-free, honest headline on imbalanced data): "
      f"{average_precision_score(y_test, probs):.3f}\n")
 
rows = []
for t in np.arange(0.10, 0.91, 0.05):
    y_hat = (probs >= t).astype(int)
    # Per-class metrics; index [1] = pathogenic (the positive class).
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, y_hat, labels=[0, 1], zero_division=0
    )
    rows.append({
        "threshold": round(t, 2),
        "path_precision": round(prec[1], 3),
        "path_recall": round(rec[1], 3),
        "path_f1": round(f1[1], 3),
        "benign_recall": round(rec[0], 3),
        "flagged_frac": round(y_hat.mean(), 3),   # share predicted pathogenic
    })
 
sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))
 
# A defensible clinical operating point: lowest threshold reaching ~0.75
# pathogenic recall, and the precision cost that comes with it.
target_recall = 0.75
ok = sweep[sweep["path_recall"] >= target_recall]
if len(ok):
    chosen = ok.iloc[-1]   # highest threshold still meeting the recall floor
    print(f"\nTo catch >= {target_recall:.0%} of pathogenic variants, use "
          f"threshold ~{chosen['threshold']}: "
          f"recall {chosen['path_recall']}, precision {chosen['path_precision']} "
          f"(flags {chosen['flagged_frac']:.0%} of variants for review).")
else:
    print(f"\nNo threshold in the swept range reaches {target_recall:.0%} recall.")